# XGBoost 再学習ノートブック v5 — 残差学習モデル本番投入

| セル | 内容 |
|------|------|
| セル1 | Google Drive マウント + セットアップ |
| セル1b | GitHub→Drive データマージ（history.db最新化） |
| セル2 | src/ 強制アップデート（GitHub最新） |
| セル3 | 展開予測モデル再学習（19特徴量ペースモデル） |
| セル4 | speed_index キャッシュ再構築 + 学習データ生成 |
| セル5 | XGB再学習（通常モデル＝ベースライン） |
| セル6 | **残差学習モデル（市場コピー排除）** |
| セル7 | **通常 vs 残差 徹底比較（AUC・重要度・レース別）** |
| セル8 | **残差モデル本番切替（条件付き自動）** |
| セル9 | 統合テスト（cal_prob合計・推論動作確認） |
| セル10 | GitHub main にプッシュ（本番反映） |

### v4からの変更点
- セル7: 通常/残差モデルのレース単位AUC比較・条件帯別の優劣分析を追加
- セル8: 残差モデルが基準を満たせば自動で本番ファイルに切替（手動コピー不要）
- セル9: 切替後のengine.py推論動作確認（_XGB_RESIDUALフラグ検出テスト）

### 判定基準
- 残差AUC >= 通常AUC × 0.95 → AI独自の予測力あり → **本番切替推奨**
- 残差AUC < 通常AUC × 0.95 → 予測力の大半は市場コピー → **現行維持**

> 使い方: セル1→1b→2〜10を順に実行。セル8で切替判定が自動で行われる。

In [ ]:
# == セル1: セットアップ ===================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = '/content/drive/MyDrive/keiba_ai'
sys.path.insert(0, BASE_DIR)
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
# == セル1b: GitHub→Drive データマージ（history.db の最新化）================
import subprocess, sqlite3, shutil, os

REPO_URL = 'https://github.com/hanagenuku/keiba_ai.git'
TMP_REPO = '/tmp/keiba_gh'
DRIVE_DB = f'{BASE_DIR}/data/history.db'

# Driveのhistory.dbスキーマを最新化（sire/dam_sire/trainer_affiliation等の
# ALTER TABLEマイグレーションを安全に適用。空リストなら実データ追記はされない）。
# 2026-07-23、GitHub側（週次ワークフローが毎回save_history_db()を呼ぶため
# 自動で最新化される）とDrive側（手動でしかsave_history_db()が呼ばれない）で
# スキーマがずれ、次のマージがOperationalError（列数不一致）で失敗する事故が
# 実際に発生したための対応。
sys.path.insert(0, BASE_DIR)
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]
from src.utils.db import save_history_db
save_history_db([], base_dir=BASE_DIR)
print('✅ Drive history.db のスキーマを最新化しました')

# Drive側の現状
_drv = sqlite3.connect(DRIVE_DB)
_drv_races = _drv.execute("SELECT COUNT(*) FROM race_history").fetchone()[0]
_drv_horses = _drv.execute("SELECT COUNT(*) FROM horse_history").fetchone()[0]
_drv_max = _drv.execute("SELECT MAX(date) FROM race_history").fetchone()[0]
_drv.close()
print(f'Drive: {_drv_races}レース / {_drv_horses}出走 / 最新{_drv_max}')

# GitHubからshallow clone（history.dbだけ取得）
if os.path.exists(TMP_REPO):
    shutil.rmtree(TMP_REPO)

try:
    from google.colab import userdata
    _pat = userdata.get('GITHUB_PAT')
    _clone_url = f'https://{_pat}@github.com/hanagenuku/keiba_ai.git'
except Exception:
    _clone_url = REPO_URL

subprocess.run(
    f'git clone --depth 1 --filter=blob:none --sparse "{_clone_url}" {TMP_REPO}',
    shell=True, capture_output=True)
subprocess.run(
    f'git -C {TMP_REPO} sparse-checkout set data/history.db',
    shell=True, capture_output=True)
subprocess.run(
    f'git -C {TMP_REPO} checkout',
    shell=True, capture_output=True)

GH_DB = f'{TMP_REPO}/data/history.db'
if not os.path.exists(GH_DB) or os.path.getsize(GH_DB) < 1000:
    print('!! GitHub DB取得失敗（LFSポインタの可能性）。スキップします。')
    print('   Drive の history.db をそのまま使用します。')
else:
    _gh = sqlite3.connect(GH_DB)
    _gh_races = _gh.execute("SELECT COUNT(*) FROM race_history").fetchone()[0]
    _gh_max = _gh.execute("SELECT MAX(date) FROM race_history").fetchone()[0]
    print(f'GitHub: {_gh_races}レース / 最新{_gh_max}')

    _drv = sqlite3.connect(DRIVE_DB)
    _drv.execute(f"ATTACH DATABASE '{GH_DB}' AS gh")

    # SELECT * だとDrive/GitHub間でスキーマ（列数・順序）がずれた場合に
    # OperationalError で落ちるため、両テーブルに共通する列名のみを
    # 明示的に指定してマージする（列名ベースなので順序差にも強い）。
    def _common_columns(conn, table):
        drv_cols = {r[1] for r in conn.execute(f'PRAGMA table_info({table})')}
        gh_cols = {r[1] for r in conn.execute(f'PRAGMA gh.table_info({table})')}
        missing_in_drive = gh_cols - drv_cols
        if missing_in_drive:
            print(f'  ⚠ {table}: GitHub側にのみ存在する列 {sorted(missing_in_drive)}'
                  f'（Drive側のスキーマ更新が必要な可能性。マージからは除外）')
        return sorted(drv_cols & gh_cols)

    race_cols = _common_columns(_drv, 'race_history')
    _drv.execute(f"""
        INSERT OR IGNORE INTO race_history ({','.join(race_cols)})
        SELECT {','.join(race_cols)} FROM gh.race_history
        WHERE race_id NOT IN (SELECT race_id FROM race_history)
    """)
    r_added = _drv.execute("SELECT changes()").fetchone()[0]

    horse_cols = _common_columns(_drv, 'horse_history')
    _drv.execute(f"""
        INSERT OR IGNORE INTO horse_history ({','.join(horse_cols)})
        SELECT {','.join(horse_cols)} FROM gh.horse_history
        WHERE race_id NOT IN (SELECT DISTINCT race_id FROM horse_history)
    """)
    h_added = _drv.execute("SELECT changes()").fetchone()[0]

    _drv.commit()
    _drv.execute("DETACH DATABASE gh")

    _new_races = _drv.execute("SELECT COUNT(*) FROM race_history").fetchone()[0]
    _new_horses = _drv.execute("SELECT COUNT(*) FROM horse_history").fetchone()[0]
    _new_max = _drv.execute("SELECT MAX(date) FROM race_history").fetchone()[0]
    _drv.close()
    _gh.close()

    print(f'\nマージ結果:')
    print(f'  レース: {_drv_races} → {_new_races} (+{r_added})')
    print(f'  出走:   {_drv_horses} → {_new_horses} (+{h_added})')
    print(f'  最新:   {_drv_max} → {_new_max}')

    if r_added == 0:
        print('\n既に最新です。')
    else:
        print(f'\n✅ {r_added}レース追加完了')

shutil.rmtree(TMP_REPO, ignore_errors=True)

In [ ]:
# == セル2: src/ 強制アップデート（GitHub最新コードを取得）==================
import urllib.request, time as _time
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
_cb = int(_time.time())
files = [
    'src/tools/__init__.py',
    'src/tools/tune_weights.py',
    'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py',
    'src/tools/build_training_data.py',
    'src/tools/train_xgb.py',
    'src/tools/calibrate_xgb.py',
    'src/tools/generate_style_advantage.py',
    'src/tools/train_pace_model.py',
    'src/tools/shap_diagnosis.py',
    'src/features/engine.py',
    'src/features/speed_index.py',
    'src/features/horse_type.py',
    'src/features/error_tags.py',
    'src/utils/config.py',
    'src/utils/db.py',
    'src/utils/model_registry.py',
    'src/scraper/parser.py',
    'src/scraper/jra_scraper.py',
    'src/models/__init__.py',
    'src/models/calibration.py',
    'src/models/calibration_xgb.py',
    'src/models/predict.py',
    'src/betting/__init__.py',
    'src/betting/make_bets.py',
    'src/betting/ev_filter.py',
    'src/betting/app_json.py',
    'src/betting/race_simulator.py',
    'src/betting/ev_calculator.py',
    'src/betting/rating_calibration.py',
    'src/betting/payout_estimator.py',
    'src/betting/dual_model.py',
    'src/betting/bet_optimizer.py',
    'src/betting/shadow.py',
]
_failed = []
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    for _retry in range(3):
        try:
            urllib.request.urlretrieve(f'{BASE_URL}/{rel}?nocache={_cb}', dest)
            print(f'OK {rel}')
            break
        except Exception as _e:
            if _retry < 2:
                _time.sleep(2 ** _retry)
            else:
                print(f'FAIL {rel}: {_e}')
                _failed.append(rel)

for rel_data in ['data/course_profiles.json', 'data/note_schema.json',
                 'data/course_distance_profiles.json']:
    dest = f'{BASE_DIR}/{rel_data}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        urllib.request.urlretrieve(f'{BASE_URL}/{rel_data}?nocache={_cb}', dest)
        print(f'OK {rel_data}')
    except Exception as e:
        print(f'SKIP {rel_data}: {e}')

if _failed:
    print(f'\n!! {len(_failed)}件失敗: {_failed}')
else:
    print()

for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

with open(f'{BASE_DIR}/src/features/engine.py') as _f:
    _eng_src = _f.read()
for _kw in ['f_speed_fig_last', 'f_popularity', '_XGB_RESIDUAL',
            '_build_pace_features_for_inference', 'error_tags',
            'calc_course_distance_features']:
    _ok = _kw in _eng_src
    print(f'  {"OK" if _ok else "NG"} engine.py: {_kw}')

print('\ndone')

In [ ]:
# == セル3: 展開予測モデル再学習（19特徴量ペースモデル）====================
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_pace_model import train_pace_model

pace_result = train_pace_model(BASE_DIR)
print(f'\n== ペースモデル結果 ==')
print(f'精度: {pace_result.get("accuracy", "N/A")}')
print(f'分布: {pace_result.get("distribution", "N/A")}')
print(f'特徴量数: {pace_result.get("n_features", "N/A")}')

In [ ]:
# == セル4: speed_index キャッシュ再構築 + 学習データ生成 ===================
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.features.speed_index import rebuild_speed_index_cache
rebuild_speed_index_cache(BASE_DIR)

for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.generate_style_advantage import generate_style_advantage
generate_style_advantage(BASE_DIR)

from src.tools.build_training_data import build_training_data
df_all = build_training_data(BASE_DIR)

print(f'\n総列数: {len(df_all.columns)}  総行数: {len(df_all)}')
print(f'f_popularity充足率: {100 * df_all["f_popularity"].notna().sum() / len(df_all):.1f}%')

In [ ]:
# == セル5: XGB再学習（通常モデル＝ベースライン）===========================
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

import pandas as _pd
from datetime import datetime as _dt, timedelta as _td

_csv_path = f'{BASE_DIR}/data/horse_features.csv'
_df_dates = _pd.read_csv(_csv_path, usecols=['date'])
_df_dates['date_obj'] = _pd.to_datetime(
    _df_dates['date'].astype(str).str.replace('-', '', regex=False).str[:8],
    format='%Y%m%d', errors='coerce'
)
_df_dates = _df_dates.dropna(subset=['date_obj'])
_max_date = _df_dates['date_obj'].max().strftime('%Y-%m-%d')
_max_dt = _dt.strptime(_max_date, '%Y-%m-%d')
_val_start_dt = _max_dt - _td(days=27)
_train_end = (_val_start_dt - _td(days=1)).strftime('%Y-%m-%d')
_val_start = _val_start_dt.strftime('%Y-%m-%d')
_val_end = _max_date
print(f'Train: 〜{_train_end}  Val: {_val_start}〜{_val_end} (4週間)')

from src.tools.train_xgb import train_xgb
from src.tools.calibrate_xgb import run_xgb_calibration

metrics_normal = train_xgb(BASE_DIR,
    train_end=_train_end,
    val_start=_val_start,
    val_end=_val_end)

normal_auc = metrics_normal.get('auc', 0)
print(f'\n通常モデル AUC: {normal_auc:.4f}')

if normal_auc >= 0.75:
    run_xgb_calibration(BASE_DIR)
    print('OK キャリブレーション完了')

In [ ]:
# == セル6: 残差学習モデル（市場コピー排除）================================
# f_popularity を特徴量から除外し、logit(p_market) を base_margin として渡す。
# モデルは「市場からのズレ」だけを学習 → AI独自の予測力を測定。
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_xgb import train_xgb

metrics_residual = train_xgb(BASE_DIR,
    train_end=_train_end,
    val_start=_val_start,
    val_end=_val_end,
    residual=True)

residual_auc = metrics_residual.get('auc', 0)
print(f'\n== 結果比較 ==')
print(f'通常AUC:   {normal_auc:.4f}')
print(f'残差AUC:   {residual_auc:.4f}')
print(f'差分:      {residual_auc - normal_auc:+.4f}')
print(f'維持率:    {residual_auc / max(normal_auc, 0.01) * 100:.1f}%')

if residual_auc >= normal_auc * 0.95:
    print('\n>>> AI独自の予測力あり。セル8で本番切替可能')
else:
    print('\n>>> 予測力の大半は市場コピー。現行モデル維持を推奨')

In [ ]:
# == セル7: 通常 vs 残差 徹底比較 ==========================================
import pickle, json, numpy as np, pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# --- モデルとデータのロード ---
with open(f'{BASE_DIR}/data/xgb_fukusho_model.pkl', 'rb') as f:
    model_normal = pickle.load(f)
with open(f'{BASE_DIR}/data/xgb_feature_cols.json') as f:
    meta_normal = json.load(f)
cols_normal = meta_normal['feature_cols']

model_residual = xgb.Booster()
model_residual.load_model(f'{BASE_DIR}/data/xgb_fukusho_model_residual.pkl')
with open(f'{BASE_DIR}/data/xgb_feature_cols_residual.json') as f:
    meta_residual = json.load(f)
cols_residual = meta_residual['feature_cols']

# val データ
df = pd.read_csv(f'{BASE_DIR}/data/horse_features.csv')
df['date_obj'] = pd.to_datetime(
    df['date'].astype(str).str.replace('-', '', regex=False).str[:8],
    format='%Y%m%d', errors='coerce')
df = df.dropna(subset=['date_obj'])
val = df[(df['date_obj'] >= pd.Timestamp(_val_start)) &
         (df['date_obj'] <= pd.Timestamp(_val_end))].copy()

# --- 通常モデル予測 ---
X_normal = val[cols_normal].reindex(columns=cols_normal, fill_value=5.0).fillna(5.0)
prob_normal = model_normal.predict_proba(X_normal)[:, 1]

# --- 残差モデル予測（base_margin付き） ---
from src.tools.train_xgb import _popularity_to_base_margin
val['_n_horses'] = val.groupby('race_id')['horse_num'].transform('count')
pop = val['f_popularity'].fillna(val['_n_horses'] / 2)
bm = _popularity_to_base_margin(pop, val['_n_horses'])

X_residual = val[cols_residual].reindex(columns=cols_residual, fill_value=5.0).fillna(5.0)
d_res = xgb.DMatrix(X_residual, feature_names=cols_residual)
d_res.set_base_margin(bm)
margin_res = model_residual.predict(d_res)
prob_residual = 1 / (1 + np.exp(-margin_res))

y = val['is_fukusho'].values

# --- 全体AUC ---
print('== 全体AUC ==')
print(f'  通常:  {roc_auc_score(y, prob_normal):.4f}')
print(f'  残差:  {roc_auc_score(y, prob_residual):.4f}')

# --- レース単位AUC比較 ---
race_results = []
for rid, grp in val.groupby('race_id'):
    idx = grp.index
    y_r = y[val.index.get_indexer(idx)]
    if y_r.sum() == 0 or y_r.sum() == len(y_r):
        continue
    pn = prob_normal[val.index.get_indexer(idx)]
    pr = prob_residual[val.index.get_indexer(idx)]
    auc_n = roc_auc_score(y_r, pn)
    auc_r = roc_auc_score(y_r, pr)
    race_results.append({
        'race_id': rid,
        'n_horses': len(grp),
        'auc_normal': auc_n,
        'auc_residual': auc_r,
        'residual_wins': auc_r > auc_n,
    })

df_race = pd.DataFrame(race_results)
n_res_wins = df_race['residual_wins'].sum()
n_total = len(df_race)
print(f'\n== レース単位 ==')
print(f'  残差が勝つレース: {n_res_wins}/{n_total} ({100*n_res_wins/max(n_total,1):.1f}%)')
print(f'  平均AUC差: {(df_race["auc_residual"] - df_race["auc_normal"]).mean():+.4f}')

# --- 特徴量重要度比較 ---
print('\n== 通常モデル 重要度 Top10 ==')
imp_normal = sorted(zip(cols_normal, model_normal.feature_importances_),
                    key=lambda x: x[1], reverse=True)[:10]
for name, imp in imp_normal:
    print(f'  {name:<40} {imp*100:5.2f}%')

print('\n== 残差モデル 重要度 Top10 ==')
imp_res_raw = model_residual.get_score(importance_type='gain')
imp_res_total = sum(imp_res_raw.values()) or 1
imp_res = sorted(imp_res_raw.items(), key=lambda x: x[1], reverse=True)[:10]
for name, imp in imp_res:
    print(f'  {name:<40} {imp/imp_res_total*100:5.2f}%')

# f_popularity が消えたか確認
if 'f_popularity' not in [n for n, _ in imp_res]:
    print('\n>>> f_popularity が残差モデルから排除されていることを確認')
else:
    print('\n!! f_popularity がまだ残っている（異常）')

In [ ]:
# == セル8: 残差モデル本番切替 =============================================
# 条件: 残差AUC >= 通常AUC × 0.95
# 切替: _residual ファイルを通常ファイル名にコピー
# → engine.py が xgb_feature_cols.json の "residual": true を検出し、
#   推論時に自動で base_margin を適用する
import shutil, json, os

THRESHOLD = 0.95
do_switch = residual_auc >= normal_auc * THRESHOLD

print(f'== 切替判定 ==')
print(f'  通常AUC:     {normal_auc:.4f}')
print(f'  残差AUC:     {residual_auc:.4f}')
print(f'  維持率:      {residual_auc / max(normal_auc, 0.01) * 100:.1f}% (基準: {THRESHOLD*100}%)')
print(f'  判定:        {"切替実行" if do_switch else "現行維持"}')

if not do_switch:
    print('\n現行モデルを維持します。残差モデルのAUCが基準未満です。')
    print('市場コピー（f_popularity）を除くとAIの予測力が大幅に低下 = 独自エッジが弱い')
    print('\n→ セル10で通常モデルのみpushしてください')
else:
    # 現行モデルをバックアップ
    for f in ['xgb_fukusho_model.pkl', 'xgb_feature_cols.json', 'xgb_calibrator.pkl']:
        src = f'{BASE_DIR}/data/{f}'
        bak = f'{BASE_DIR}/data/{f}.bak_before_residual'
        if os.path.exists(src):
            shutil.copy2(src, bak)
            print(f'  バックアップ: {f} → {f}.bak_before_residual')

    # 残差モデルを本番ファイルにコピー
    shutil.copy2(
        f'{BASE_DIR}/data/xgb_fukusho_model_residual.pkl',
        f'{BASE_DIR}/data/xgb_fukusho_model.pkl'
    )
    shutil.copy2(
        f'{BASE_DIR}/data/xgb_feature_cols_residual.json',
        f'{BASE_DIR}/data/xgb_feature_cols.json'
    )
    print('\n  xgb_fukusho_model_residual.pkl → xgb_fukusho_model.pkl')
    print('  xgb_feature_cols_residual.json → xgb_feature_cols.json')

    # 確認: residualフラグが入っているか
    with open(f'{BASE_DIR}/data/xgb_feature_cols.json') as f:
        check = json.load(f)
    assert check.get('residual') == True, 'residualフラグがない！'
    print(f'\n  residualフラグ: {check["residual"]}  特徴量数: {len(check["feature_cols"])}')

    # 残差モデル用キャリブレーション再実行
    print('\n== 残差モデル用キャリブレーション ==')
    for key in list(sys.modules.keys()):
        if 'src' in key:
            del sys.modules[key]
    from src.tools.calibrate_xgb import run_xgb_calibration
    run_xgb_calibration(BASE_DIR)

    print('\n>>> 残差モデルに切替完了。セル9で動作確認 → セル10でpush')

In [ ]:
# == セル9: 統合テスト =====================================================
import pickle, json, numpy as np, pandas as pd

# モジュールキャッシュクリア（切替後のモデルを再ロードするため）
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# --- 基本情報 ---
with open(f'{BASE_DIR}/data/xgb_feature_cols.json') as f:
    meta = json.load(f)
cols = meta['feature_cols']
is_residual = meta.get('residual', False)

print(f'モデル特徴量数: {len(cols)}')
print(f'学習日時: {meta.get("trained_at", "不明")}')
print(f'Val AUC: {meta.get("val_auc", "不明")}')
print(f'残差モード: {is_residual}')

# --- engine.py が _XGB_RESIDUAL を正しく検出するか ---
from src.features.engine import init_engine, _XGB_RESIDUAL
init_engine(BASE_DIR)
from src.features.engine import _XGB_RESIDUAL
print(f'\nengine._XGB_RESIDUAL = {_XGB_RESIDUAL}')
if is_residual:
    assert _XGB_RESIDUAL == True, '_XGB_RESIDUAL が True になっていない！'
    print('  → 残差モード検出OK。推論時にbase_marginが自動適用される')
else:
    assert _XGB_RESIDUAL == False, '_XGB_RESIDUAL が False になっていない！'
    print('  → 通常モード')

# --- cal_prob 合計確認 ---
with open(f'{BASE_DIR}/data/xgb_calibrator.pkl', 'rb') as f:
    calib = pickle.load(f)

csv = pd.read_csv(f'{BASE_DIR}/data/horse_features.csv')
csv['date_obj'] = pd.to_datetime(
    csv['date'].astype(str).str.replace('-', '', regex=False).str[:8],
    format='%Y%m%d', errors='coerce')
csv = csv.dropna(subset=['date_obj'])
val_data = csv[csv['date_obj'] >= pd.Timestamp(_val_start)].copy()

if len(val_data) > 0:
    sums = []
    if is_residual:
        import xgboost as xgb
        from src.tools.train_xgb import _popularity_to_base_margin
        model_r = xgb.Booster()
        model_r.load_model(f'{BASE_DIR}/data/xgb_fukusho_model.pkl')
        val_data['_n_horses'] = val_data.groupby('race_id')['horse_num'].transform('count')
    else:
        with open(f'{BASE_DIR}/data/xgb_fukusho_model.pkl', 'rb') as f:
            model_n = pickle.load(f)

    for rid, grp in val_data.groupby('race_id'):
        X = grp[cols].reindex(columns=cols, fill_value=5.0).fillna(5.0)
        if is_residual:
            pop = grp['f_popularity'].fillna(grp['_n_horses'] / 2)
            bm = _popularity_to_base_margin(pop, grp['_n_horses'])
            dm = xgb.DMatrix(X, feature_names=cols)
            dm.set_base_margin(bm)
            margin = model_r.predict(dm)
            raw = 1 / (1 + np.exp(-margin))
        else:
            raw = model_n.predict_proba(X)[:, 1]
        cal = np.array(calib.transform(raw))
        sums.append(cal.sum())

    sums = np.array(sums)
    print(f'\n== cal_prob合計（理論≒3.0）==')
    print(f'  レース数: {len(sums)}')
    print(f'  平均: {sums.mean():.3f}')
    print(f'  中央値: {np.median(sums):.3f}')
    print(f'  範囲: [{sums.min():.3f}, {sums.max():.3f}]')
    ok = 2.0 <= sums.mean() <= 4.0
    print(f'  チェック: {"OK" if ok else "NG（要確認）"}')

print('\n>>> 統合テスト完了')

In [ ]:
# == セル10: GitHub main にプッシュ =========================================
import requests, base64, json as _json, os as _os
from google.colab import userdata

GITHUB_PAT = userdata.get('GITHUB_PAT')
REPO = 'hanagenuku/keiba_ai'

FILES = [
    'data/xgb_fukusho_model.pkl',
    'data/xgb_feature_cols.json',
    'data/xgb_calibrator.pkl',
    'data/pace_model.pkl',
    'data/jockey_pace_stats.json',
    'data/speed_index_cache.pkl',
    'data/xgb_fukusho_model_residual.pkl',
    'data/xgb_feature_cols_residual.json',
]

def push_file(relpath, pat, message):
    url = f'https://api.github.com/repos/{REPO}/contents/{relpath}'
    h = {'Authorization': f'token {pat}', 'Accept': 'application/vnd.github.v3+json'}
    r = requests.get(url, headers=h, params={'ref': 'main'})
    sha = r.json().get('sha') if r.status_code == 200 else None
    with open(f'{BASE_DIR}/{relpath}', 'rb') as f:
        content = base64.b64encode(f.read()).decode()
    payload = {'message': message, 'content': content, 'branch': 'main'}
    if sha:
        payload['sha'] = sha
    r = requests.put(url, headers=h, json=payload)
    ok = r.status_code in (200, 201)
    status = 'OK' if ok else f'NG({r.status_code})'
    print(f'{status} {relpath}')
    if not ok:
        print(f'  -> {r.json().get("message", "")}')
    return ok

with open(f'{BASE_DIR}/data/xgb_feature_cols.json') as f:
    _meta = _json.load(f)
n = len(_meta['feature_cols'])
auc_val = _meta.get('val_auc', 0)
is_res = _meta.get('residual', False)
mode = 'residual' if is_res else 'normal'

print(f'=== push前確認 ===')
print(f'  モード:    {mode}')
print(f'  特徴量数:  {n}')
print(f'  Val AUC:   {auc_val:.4f}')

msg = f'model: retrain v5 {n}feat AUC={auc_val:.4f} ({mode})'
print(f'\nコミットメッセージ: {msg}\n')

pushed = 0
for rel in FILES:
    if not _os.path.exists(f'{BASE_DIR}/{rel}'):
        print(f'SKIP（未生成）: {rel}')
        continue
    if push_file(rel, GITHUB_PAT, msg):
        pushed += 1

print(f'\n完了: {pushed}/{len(FILES)} ファイルをpush')
if is_res:
    print('\n>>> 残差モデルが本番に反映されました。')
    print('    次回の週末ワークフローからAI独自予測で買い目が生成されます。')
    print('    数週後にフォワードROIを確認し、エッジの有無を判定してください。')